# HU-03 · Carrera multijugador

[Inicio](../../tests.ipynb) · Anterior: [HU-02 · Práctica individual](hu02_practica.ipynb) · Siguiente: [HU-04 · Ranking mensual](hu04_ranking.ipynb)

Las fichas incluyen los criterios del Plan Maestro. Las referencias U/G/F/W/D y las rutas abreviadas en **Evidencia** se interpretan en la [matriz maestra](../../docs/test/qa-master-matrix.md). Una ejecución local no implica un resultado publicado ni prueba un control de entrega.

## TC-FUNC-06

- **Resumen de la ficha:** Inicio sincronizado de la carrera (cuenta regresiva de 3 s y mismo texto)
- **Método previsto:** Script WS.
- **Estado:** **V — Verificado**.
- **Evidencia:** En la matriz (**V**): `frontend/e2e/race-start.feature`: en Chromium local, ambos clientes vieron `3 → 2 → 1` antes del mismo texto. Intervalos observados: 2991,782 y 2998,465 ms. Desfase de recepción de `race_start`: 0,076 ms (una muestra; no mide desfase de renderizado ni aplica un umbral de aceptación). [Fuente de prueba](../../frontend/e2e/race-start.feature).
- **Brecha o límite:** El desfase de recepción no mide el desfase de renderizado ni define un umbral de sincronía.

**Criterio de aceptación literal** · Plan §5.5 → [HU-03 #10](https://github.com/1Bnja/Typr/issues/10).

```gherkin
Característica: Carrera multijugador en vivo

  Escenario: Inicio sincronizado de la carrera
    Dado que hay al menos 2 jugadores conectados y listos en la sala
    Cuando el anfitrión presiona el botón para iniciar la carrera
    Entonces el sistema muestra una cuenta regresiva de 3 segundos para todos los participantes
    Y al finalizar el conteo, despliega simultáneamente el mismo texto a cada jugador
```

**Gherkin ejecutable (original en inglés)** · [Fuente](../../frontend/e2e/race-start.feature).

```gherkin
@TC-FUNC-06
Feature: Starting a multiplayer race

  Scenario: Both players count down three seconds before receiving the same text
    Given I register a new account in the browser
    And two players are ready to start a race in separate browser contexts
    When the host starts the race
    Then both players see a three-second countdown followed by the same race text
```

## TC-FUNC-07

- **Resumen de la ficha:** Un error de escritura impide avanzar hasta corregirlo.
- **Método previsto:** Manual + E2E.
- **Estado:** **V — Verificado**.
- **Evidencia:** En la matriz (**P**): `rooms.test.mjs` (**F**) prueba bloqueo y corrección en lógica; `test_ws_rooms.py` (**U**) rechaza progreso inválido. Falta el flujo de escritura multijugador en navegador.
  
  Evidencia local posterior o adicional: Chromium visible: 1 escenario y 12 pasos correctos (7,449 s), sin reintentos; se observó bloqueo visual y de progreso, corrección y avance sincronizado. El intento anterior movía el cursor 15,8 px pese a no avanzar. Registro Engram #1022. [Fuente de prueba](../../frontend/e2e/room-typing.feature).
- **Brecha o límite:** No se repitió en otros motores.

**Criterio de aceptación literal** · Plan §5.5 → [HU-03 #10](https://github.com/1Bnja/Typr/issues/10).

```gherkin
Característica: Carrera multijugador en vivo

  Escenario: Bloqueo de avance por error tipográfico
    Dado que un jugador está disputando una carrera
    Y comete un error en el carácter actual
    Cuando intenta tipear el siguiente carácter sin borrar la falla
    Entonces el sistema bloquea el avance de su cursor en el texto
    Y mantiene congelado su porcentaje de progreso hasta corregir el error
```

**Gherkin ejecutable (original en inglés)** · [Fuente](../../frontend/e2e/room-typing.feature).

```gherkin
@TC-FUNC-07 @room_typing
Feature: Typing mistakes in a multiplayer race

  Scenario: A mistake blocks the cursor and both progress displays until corrected
    Given I register a new account in the browser
    And I start a race with a rival in a separate browser context
    When I type a wrong character
    Then the wrong character is highlighted without moving my caret or increasing either progress display
    When I try the expected character before Backspace
    Then the same mistake still blocks typing and progress
    When I erase the mistake without typing a valid character
    Then the block clears without increasing progress
    When I type the next valid character
    Then both progress displays advance and my cursor advances only for valid input
```

## TC-FUNC-08

- **Resumen de la ficha:** La carrera finaliza y entrega resultados tanto si todos completan el texto como si se agota el tiempo límite.
- **Método previsto:** Script WS.
- **Estado:** **V — Verificado**.
- **Evidencia:** En la matriz (**V**): [test_ws_rooms.py](../../backend/tests/test_ws_rooms.py) cubre el final por todos los jugadores o al vencer el plazo. Para el puntaje proporcional, [test_race_scoring.py](../../backend/tests/test_race_scoring.py) comprueba el valor exacto en lógica; el caso WebSocket del plazo comprueba un puntaje positivo y el progreso parcial, no la fórmula exacta.
- **Brecha o límite:** Se comprueban eventos y resultados de servidor, no la presentación de resultados en navegador.

**Criterio de aceptación literal** · Plan §5.5 → [HU-03 #10](https://github.com/1Bnja/Typr/issues/10).

```gherkin
Característica: Carrera multijugador en vivo

  Esquema del escenario: Finalización de la carrera y entrega de resultados
    Dado que la carrera multijugador está en curso
    Cuando ocurre la condición de fin "<condicion_termino>"
    Entonces el sistema da por concluida la partida
    Y presenta a cada jugador su tiempo total, precisión, posición y puntaje calculado
    Ejemplos:
      | condicion_termino                     |
      | todos los jugadores completan el texto |
      | se agota el tiempo límite de la sala   |
```

## TC-FUNC-09

- **Resumen de la ficha:** Cada jugador tiene su propio resultado registrado y su historial muestra las métricas que le corresponden.
- **Método previsto:** API.
- **Estado:** **V — Verificado**.
- **Evidencia:** En la matriz (**V**): `integration/test_rooms_postgresql.py` (**U**) verifica que ambos jugadores consultan `GET /api/v1/users/me/stats` con sus propias credenciales y recuperan sus métricas persistidas para el mismo `gameId`, con posición y precisión distintas.
  
  Evidencia local posterior o adicional: Una prueba focalizada en PostgreSQL desechable con Alembic 0006 pasó en 3,842 s (Engram #1026); cada token consultó sus propias métricas del mismo `gameId`. [Fuente de prueba](../../backend/tests/integration/test_rooms_postgresql.py).
- **Brecha o límite:** No se usó el evento compartido `race_end` como prueba de aislamiento.

**Criterio de aceptación literal** · Plan §5.5 → [HU-03 #10](https://github.com/1Bnja/Typr/issues/10).

```gherkin
Característica: Carrera multijugador en vivo

  Escenario: Registro separado de los resultados por jugador
    Dado que la carrera multijugador ha finalizado
    Cuando el sistema registra la partida
    Entonces crea un resultado independiente para cada jugador con su propio tiempo, precisión, posición y puntaje
    Y asocia cada resultado únicamente al perfil de ese jugador
    Y marca la partida como modo multijugador, con su sala asociada
```

**Gherkin ejecutable (original en inglés)** · [Fuente](../../backend/tests/features/participation.feature).

```gherkin
Feature: Completed matches preserve participation integrity

  @RI-04 @RI-05
  Scenario: A completed race stores exactly one participation per player
    Given a live race with two signed-in members
    When both members complete the race
    Then the game is linked to the live room and has one participation per player
    And PostgreSQL rejects a duplicate game and user pair without adding a row
```

El Gherkin cubre la persistencia de participaciones por jugador, no toda la comprobación de métricas e historiales documentada en integración.

## Reproducción local opcional

Prepare el entorno desechable de la [guía de pruebas](../../test/README.md). Ejecute solo la celda que necesita, con el kernel iniciado en la raíz de Typr o dentro de `notebooks/`. No ejecute en paralelo. Abrir el cuaderno no inicia pruebas; la salida describe una corrida local, no cambia el estado de las fichas. Chromium se abre en una sesión gráfica local. Las celdas de backend ejecutan la suite indicada completa, no solo los casos de esta página.

### Criterios de aceptación · TC-FUNC-06 y TC-FUNC-07

**Plan (§5.5):** Los jugadores reciben el mismo texto tras una cuenta regresiva de 3 s. Un error tipográfico bloquea el avance hasta que se corrige.

**Qué comprueba esta celda:** Chromium filtra ambos casos. Dos navegadores ven 3, 2 y 1 sin texto, luego el mismo texto de carrera. Un error se resalta sin mover el cursor ni cambiar el progreso que ven el jugador y su rival; borrar el error no suma progreso y solo el carácter válido hace avanzar ambos indicadores.

**Límite:** No mide el desfase de renderizado entre jugadores ni prueba otros motores.

In [ ]:
from pathlib import Path
from runpy import run_path

directorio = Path.cwd().resolve()
for raiz in (directorio, *directorio.parents):
    if (raiz / "docker-compose.test.yml").is_file() and (raiz / "notebooks/ejecutar_qa.py").is_file():
        break
else:
    raise RuntimeError("Inicie el kernel desde la raíz de Typr o dentro de notebooks/.")

run_path(str(raiz / "notebooks/ejecutar_qa.py"))["ejecutar"]('browser-gherkin', etiquetas='@TC-FUNC-06 or @TC-FUNC-07')

### Criterios de aceptación · TC-FUNC-08 y TC-FUNC-09

**Plan (§5.5):** La carrera termina y entrega resultados cuando todos completan el texto o vence el plazo; los resultados se registran por jugador.

**Qué comprueba esta celda:** Ejecuta toda la suite de pruebas unitarias, integración y migraciones del backend. `test_ws_rooms.py` comprueba el cierre por ambas causas; `test_race_scoring.py` verifica la fórmula de puntaje proporcional por separado. `test_rooms_postgresql.py` comprueba posiciones y precisiones distintas y que cada historial devuelve una sola partida multijugador con las métricas de ese jugador.

**Límite:** Son comprobaciones de WebSocket, API y persistencia; no se comprueba la pantalla de resultados.

In [ ]:
from pathlib import Path
from runpy import run_path

directorio = Path.cwd().resolve()
for raiz in (directorio, *directorio.parents):
    if (raiz / "docker-compose.test.yml").is_file() and (raiz / "notebooks/ejecutar_qa.py").is_file():
        break
else:
    raise RuntimeError("Inicie el kernel desde la raíz de Typr o dentro de notebooks/.")

run_path(str(raiz / "notebooks/ejecutar_qa.py"))["ejecutar"]('backend-unit-integration-migrations')

[Inicio](../../tests.ipynb) · Anterior: [HU-02 · Práctica individual](hu02_practica.ipynb) · Siguiente: [HU-04 · Ranking mensual](hu04_ranking.ipynb)